# GRM vs PC-Relate kinship, coloured by round-2 PC distance between i and j
Individual pairs: notebook only, nothing saved.

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

In [ ]:
OUT = single_target(cfg, "pcrelate")
ids = pd.read_csv(OUT.replace(".bin", ".id"), header=None, dtype=str)[0]
n = len(ids)
kin = np.fromfile(OUT, dtype="float32")
grm = np.concatenate([np.fromfile(u, dtype="float32") for u, _ in target_items(cfg, "grm_shards")])
assert len(kin) == len(grm) == n * (n + 1) // 2

K2 = cfg.params.get("r2_k") or cfg.param("pcr_n_pcs")   # mixed sets have no round-2 gate
r2 = pd.read_csv(single_target(cfg, "round2_pca"), sep="\t", dtype={"id": str})
r2 = r2[r2["set"] == "cohort"].drop_duplicates("id").set_index("id")
pcs2 = [p for p in r2 if p.startswith("PC")][:K2]
X = r2.reindex(ids)[pcs2].to_numpy()
print(n, "people,", int(np.isnan(X).any(axis=1).sum()), "without round-2 scores")

In [ ]:
i, j = np.tril_indices(n, -1)   # every off-diagonal pair, same order as the triangle without its diagonal
diag = np.arange(1, n + 1) * (np.arange(1, n + 1) + 1) // 2 - 1
off = np.ones(len(kin), bool); off[diag] = False
g, k = grm[off] / 2, kin[off]
d = np.linalg.norm(X[i] - X[j], axis=1).astype("float32")
del i, j
ok = ~np.isnan(d)
g, k, d = g[ok], k[ok], d[ok]
o = np.argsort(d, kind="stable")   # far pairs drawn last
print(f"{len(d):,} pairs")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, lim in zip(axes, [(-0.06, 0.06), (-0.06, 0.55)]):
    sc = ax.scatter(g[o], k[o], c=d[o], s=0.5, cmap="viridis", alpha=0.5, linewidths=0, rasterized=True)
    ax.plot(lim, lim, "r:", lw=0.8)
    ax.set(xlim=lim, ylim=lim, xlabel="GRM / 2", ylabel="PC-Relate kinship")
axes[0].set_title(f"{RUN_ID}: bulk (all pairs)"); axes[1].set_title("with relatives")
fig.colorbar(sc, ax=axes, label=f"distance in round-2 PC space (PC1-{K2})")
plt.show()
del o

## Distributions (bins with 21+ pairs)

In [ ]:
diag = np.arange(1, n + 1) * (np.arange(1, n + 1) + 1) // 2 - 1
off = np.ones(len(kin), bool); off[diag] = False
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (lo, hi, logy) in zip(axes, [(-0.06, 0.06, False), (0.02, 0.6, True), (0.35, 0.65, True)]):
    for v, lab in [(grm[off] / 2, "GRM / 2"), (kin[off], "PC-Relate")]:
        h, e = np.histogram(v[(v > lo) & (v < hi)], bins=120)
        ax.stairs(np.where(h >= 21, h, 0), e, label=lab)
    ax.set(xlabel="kinship", title=f"{lo} to {hi}", yscale="log" if logy else "linear")
axes[0].legend(); plt.show()

## Duplicate / MZ pairs (PC-Relate kinship > 0.354): where does the extra GRM come from?
- `grm_norm` = GRM_ij / sqrt(GRM_ii GRM_jj): ~1 means the excess is just inflated diagonals (1 + F)
- `grm_ii`, `grm_jj`: GRM self-relatedness (1 + F against the cohort's frequencies)
- `pcr_self`: PC-Relate 2 x self-kinship - 1 (F with ancestry-specific frequencies)
- `pc_centroid_dist`: mean distance of i and j from the cohort centre in round-2 PCs (ancestry outliers)
- `pc_pair_dist`: distance between i and j (should be ~0 for a true duplicate)
- `het_ratio`: GRM_ii / PC-Relate self, high when F is ancestry-driven rather than true inbreeding

In [ ]:
dup = np.flatnonzero(off & (kin > 0.354))
di = ((np.sqrt(8 * dup.astype(float) + 1) - 1) // 2).astype(np.int64); dj = dup - di * (di + 1) // 2
gd, kd = grm[diag], 2 * kin[diag]
cent = np.nanmedian(X, axis=0)
cdist = np.linalg.norm(X - cent, axis=1)
D = pd.DataFrame({"grm_half": grm[dup] / 2, "pcrelate": kin[dup],
                  "grm_norm": grm[dup] / np.sqrt(gd[di] * gd[dj]),
                  "grm_ii": gd[di], "grm_jj": gd[dj],
                  "pcr_self_i": kd[di] - 1, "pcr_self_j": kd[dj] - 1,
                  "pc_centroid_dist": (cdist[di] + cdist[dj]) / 2,
                  "pc_pair_dist": np.linalg.norm(X[di] - X[dj], axis=1)})
D["het_ratio"] = (D.grm_ii + D.grm_jj) / (D.pcr_self_i + D.pcr_self_j + 2)
D["cluster"] = np.where(D.grm_half - D.pcrelate > 0.05, "GRM too high", "on y = x")
print(len(D), "pairs")
D.groupby("cluster").median().T

In [ ]:
cols = ["grm_norm", "grm_ii", "pcr_self_i", "pc_centroid_dist", "pc_pair_dist", "het_ratio"]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, c in zip(axes.flat, cols):
    sc = ax.scatter(D.pcrelate, D.grm_half, c=D[c], s=8, cmap="viridis")
    ax.plot([0.35, 0.7], [0.35, 0.7], "r:", lw=0.8)
    ax.set(xlabel="PC-Relate kinship", ylabel="GRM / 2", title=c); fig.colorbar(sc, ax=ax)
plt.tight_layout(); plt.show()